In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim import Adam
import numpy as np
import dgl
import dgl.data
from dgl.data import load_data

In [3]:
dataset = dgl.data.CoraGraphDataset()
g = dataset[0]



  NumNodes: 2708
  NumEdges: 10556
  NumFeats: 1433
  NumClasses: 7
  NumTrainingSamples: 140
  NumValidationSamples: 500
  NumTestSamples: 1000
Done loading data from cached files.


In [ ]:
from dgl.nn import SAGEConv

class GraphSAGE(nn.Module):
    def __init__(self, in_feats, h_feats):
        super(GraphSAGE, self).__init__()
        
        self.in_feats = in_feats
        self.h_feats = h_feats
        
        self.conv1 = SAGEConv(self.in_feats, self.h_feats, aggregator_type = 'mean')
        self.conv2 = SAGEConv(self.h_feats, self.h_feats, aggregator_type = 'mean')
        self.relu = nn.ReLU()
        
    def forward(self, g, in_feats):
        h = self.conv1(g, in_feats)
        h = self.relu(h)
        h = self.conv2(g, h)
        return h

In [5]:
features = g.ndata['feat']
labels = g.ndata['label']
train_mask = g.ndata['train_mask']
val_mask = g.ndata['val_mask']
test_mask = g.ndata['test_mask']

cora dataset contains number of 2704 data
each data have feature dim 1433


In [7]:
print(features.shape)

torch.Size([2708, 1433])


In [10]:
dataset.num_classes

7

In [8]:
labels.shape

torch.Size([2708])

In [9]:
val_mask.shape

torch.Size([2708])

In [4]:
#cora

import dgl
import torch
import torch.nn as nn
import torch.nn.functional as F
from dgl.nn import SAGEConv
from dgl.data import CoraGraphDataset

# Load the Cora dataset
dataset = CoraGraphDataset()
g = dataset[0]

# Define the GraphSAGE model
class GraphSAGE(nn.Module):
    def __init__(self, in_feats, h_feats, num_classes):
        super(GraphSAGE, self).__init__()
        self.conv1 = SAGEConv(in_feats, h_feats, 'mean')
        self.conv2 = SAGEConv(h_feats, num_classes, 'mean')

    def forward(self, g, features):
        h = self.conv1(g, features) #(in_feats, h_feats)
        h = F.relu(h)
        h = self.conv2(g, h)
        return h

# Prepare features, labels, and masks
features = g.ndata['feat']
labels = g.ndata['label']
train_mask = g.ndata['train_mask']
val_mask = g.ndata['val_mask']
test_mask = g.ndata['test_mask']

# Initialize the model
model = GraphSAGE(in_feats=features.shape[1], h_feats=16, num_classes=dataset.num_classes)

# Define the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training loop
def train(num_epochs):
    for epoch in range(num_epochs):
        model.train()
        logits = model(g, features)
        loss = F.cross_entropy(logits[train_mask], labels[train_mask])

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_acc = (logits[train_mask].argmax(dim=1) == labels[train_mask]).float().mean()
        val_acc = (logits[val_mask].argmax(dim=1) == labels[val_mask]).float().mean()
        print(f'Epoch {epoch}, Loss: {loss.item():.4f}, Train Acc: {train_acc.item():.4f}, Val Acc: {val_acc.item():.4f}')

train(num_epochs=100)


  NumNodes: 2708
  NumEdges: 10556
  NumFeats: 1433
  NumClasses: 7
  NumTrainingSamples: 140
  NumValidationSamples: 500
  NumTestSamples: 1000
Done loading data from cached files.
Epoch 0, Loss: 1.9600, Train Acc: 0.1429, Val Acc: 0.0700
Epoch 1, Loss: 1.9203, Train Acc: 0.1571, Val Acc: 0.0700
Epoch 2, Loss: 1.8765, Train Acc: 0.1786, Val Acc: 0.0760
Epoch 3, Loss: 1.8263, Train Acc: 0.2571, Val Acc: 0.0820
Epoch 4, Loss: 1.7715, Train Acc: 0.4786, Val Acc: 0.0980
Epoch 5, Loss: 1.7139, Train Acc: 0.6500, Val Acc: 0.1540
Epoch 6, Loss: 1.6535, Train Acc: 0.7857, Val Acc: 0.3000
Epoch 7, Loss: 1.5907, Train Acc: 0.9286, Val Acc: 0.3940
Epoch 8, Loss: 1.5256, Train Acc: 0.9714, Val Acc: 0.4660
Epoch 9, Loss: 1.4589, Train Acc: 0.9786, Val Acc: 0.5120
Epoch 10, Loss: 1.3904, Train Acc: 0.9857, Val Acc: 0.5480
Epoch 11, Loss: 1.3200, Train Acc: 0.9857, Val Acc: 0.5560
Epoch 12, Loss: 1.2480, Train Acc: 0.9929, Val Acc: 0.5740
Epoch 13, Loss: 1.1753, Train Acc: 1.0000, Val Acc: 0.5840
Ep